Generate fixation-probability results for the graph comparisons. Configure `simEMoranFix_jar` in `jar_locations.py` before running. Existing results are not overwritten. The measured population-size and expected-carrying-capacity files used by the plots must already be in `pop_sizes/graph`.

In [ ]:
import sys
from pathlib import Path
from config import *
sys.path.insert(0, str(next(p for p in Path(folder_loc).parents if (p / 'jar_locations.py').is_file())))
from generation import generate_arguments_string, run_simulation
pop_sizes_loc = next(p for p in Path(folder_loc).parents if (p / 'jar_locations.py').is_file()) / 'pop_sizes' / 'graph'

from jar_locations import simEMoranFix_jar


In [ ]:
def generate_fixation(graph, pop_size, d, fitness):
    code = graph.replace('F', 'upper_F').replace('f', 'lower_f')
    name = f'eMoran-N-{pop_size}-geom-{code}-d-{d}-r-{fitness}-tr-{timerelax}'
    run_simulation(results_loc, name, simEMoranFix_jar, generate_arguments_string({
        'timerelax': timerelax,
        'module': 'eMoran',
        'digits': 6,
        'delay': 0,
        'timestep': 1,
        'popsize': pop_size,
        'popupdate': 'e',
        'geometry': graph,
        'fitness': f'1,{fitness}',
        'mutation': 0,
        'deathrate': d,
        'init': ('mutant', '1,0'),
        'monostop': None,
        'samples': samples,
    }))


In [ ]:
pop_sizes = pop_sizes_loc
jobs = set()
for d in (0.2, 0.5, 0.6):
    jobs.add(('M', round(K / (1 - d)), d))
    for k in (3, 4, 6, 8, 12):
        pop_size = int((pop_sizes / f'pop_size_r{k}_d_{d}.txt').read_text())
        jobs.add((f'r{k}', pop_size, d))
    if d in (0.2, 0.5):
        for k in (3, 4, 6, 8):
            pop_size = int((pop_sizes / f'pop_size_f{k}_d_{d}.txt').read_text())
            jobs.add((f'f{k}', pop_size, d))
        for graph, k in (('n', 4), ('m', 8)):
            pop_size = int((pop_sizes / f'pop_size_{graph}_d_{d}.txt').read_text())
            expected_K = float((pop_sizes / f'pop_size_{graph}_d_{d}_expected_K.txt').read_text())
            regular_size = int((pop_sizes / f'pop_size_r{k}_d_{d}_main_{graph}.txt').read_text())
            jobs.update(((graph, pop_size, d), ('M', round(expected_K / (1 - d)), d),
                         (f'r{k}', regular_size, d)))
for graph, pop_size, d in sorted(jobs):
    for fitness in rs:
        generate_fixation(graph, pop_size, d, fitness)
